# LAB 1 — Multi-Query RAG com LangChain
## Aula 7 · MBA RAG & CAG Aplicados a Direito e Segurança Pública

**Objetivo:** Implementar o LangChain `MultiQueryRetriever` para gerar 4 variações semânticas de queries jurídicas, executar retrieval para cada variação e deduplicar os resultados com threshold de similaridade de coseno.

**Tempo estimado:** 40 minutos

**Pré-requisitos:**
- OpenSearch rodando (LAB1 da Aula 1)
- vLLM com Llama 3.1 8B rodando (LAB3 da Aula 1)
- Corpus jurídico indexado (LAB4 da Aula 2)

---
**Checklist de entrega:**
- [ ] MultiQueryRetriever configurado com vLLM local
- [ ] 4 variações geradas para 3 queries jurídicas distintas
- [ ] Deduplicação por similaridade de coseno implementada
- [ ] Tabela comparativa: docs com e sem deduplicação
- [ ] Análise do comportamento com N=8

## 1. Instalação de Dependências

In [ ]:
# Instalar dependências necessárias
!pip install -q langchain langchain-openai langchain-community \
    opensearch-py sentence-transformers scikit-learn pandas matplotlib

print('✅ Dependências instaladas com sucesso!')

## 2. Configuração do Ambiente

In [ ]:
import os
import json
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List

# Suprimir logs excessivos do LangChain para focar no output educacional
logging.basicConfig(level=logging.WARNING)
logging.getLogger('langchain').setLevel(logging.INFO)

# Configurações do ambiente
VLLM_BASE_URL = os.getenv('VLLM_BASE_URL', 'http://localhost:8000/v1')
OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', '9200'))
INDEX_NAME = 'corpus_juridico_aula7'
EMBEDDING_MODEL = 'BAAI/bge-m3'

print(f'vLLM URL: {VLLM_BASE_URL}')
print(f'OpenSearch: {OPENSEARCH_HOST}:{OPENSEARCH_PORT}')
print(f'Índice: {INDEX_NAME}')

## 3. Inicialização dos Clientes

In [ ]:
from langchain_openai import ChatOpenAI
from sentence_transformers import SentenceTransformer
from opensearchpy import OpenSearch

# ── LLM via vLLM ──────────────────────────────────────────────────────
llm = ChatOpenAI(
    base_url=VLLM_BASE_URL,
    api_key='dummy-key',  # vLLM local não requer autenticação
    model='meta-llama/Llama-3.1-8B-Instruct',
    temperature=0.7,       # temperatura maior = mais criatividade nas variações
    max_tokens=512
)

# ── Modelo de Embeddings ───────────────────────────────────────────────
print('Carregando modelo de embeddings BGE-M3...')
embeddings_model = SentenceTransformer(EMBEDDING_MODEL)
print(f'✅ BGE-M3 carregado | Dimensão: {embeddings_model.get_sentence_embedding_dimension()}')

# ── Cliente OpenSearch ─────────────────────────────────────────────────
os_client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=('admin', os.getenv('OPENSEARCH_PASS', 'Admin123!')),
    use_ssl=False,
    verify_certs=False
)

# Verificar conectividade
info = os_client.info()
print(f'✅ OpenSearch conectado | Versão: {info["version"]["number"]}')

## 4. Prompt Customizado para Domínio Jurídico

In [ ]:
from langchain.prompts import PromptTemplate

# ── Prompt especializado para geração de variações jurídicas ──────────
PROMPT_MULTI_QUERY = PromptTemplate(
    input_variables=['question'],
    template="""Você é um assistente jurídico especializado em recuperação de informações.
Sua tarefa é gerar exatamente 4 versões alternativas da pergunta abaixo para melhorar
a recuperação de documentos em um sistema RAG jurídico.

Produza variações que cubram:
1. Terminologia técnica jurídica (linguagem dos códigos e doutrina)
2. Linguagem coloquial (como um cidadão comum perguntaria)
3. Perspectiva processual (foco nos procedimentos e prazos)
4. Perspectiva constitucional/principiológica (direitos fundamentais envolvidos)

Retorne APENAS as 4 perguntas, uma por linha, sem numeração, sem prefixos.

Pergunta original: {question}

Versões alternativas:"""
)

# ── Teste do prompt com uma query de exemplo ──────────────────────────
query_teste = 'Posso ser preso por não pagar uma dívida?'
response = llm.invoke(PROMPT_MULTI_QUERY.format(question=query_teste))

print(f'Query original: {query_teste}')
print('\n--- Variações Geradas ---')
for i, variacao in enumerate(response.content.strip().split('\n'), 1):
    if variacao.strip():
        print(f'{i}. {variacao.strip()}')

## 5. Implementação do MultiQueryRetriever com OpenSearch

In [ ]:
# ── Retriever OpenSearch personalizado para LangChain ─────────────────
from langchain.schema import BaseRetriever, Document
from langchain.callbacks.manager import CallbackManagerForRetrieverRun

class OpenSearchKNNRetriever(BaseRetriever):
    """Retriever LangChain-compatível que usa OpenSearch kNN."""
    
    opensearch_client: object
    embeddings_model: object
    index_name: str
    k: int = 5
    
    class Config:
        arbitrary_types_allowed = True
    
    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        # Gerar embedding da query
        query_vector = self.embeddings_model.encode(
            query, normalize_embeddings=True
        ).tolist()
        
        # Busca kNN no OpenSearch
        response = self.opensearch_client.search(
            index=self.index_name,
            body={
                'size': self.k,
                'query': {
                    'knn': {
                        'embedding': {
                            'vector': query_vector,
                            'k': self.k
                        }
                    }
                },
                '_source': ['content', 'source', 'chunk_id']
            }
        )
        
        # Converter para formato LangChain
        documents = []
        for hit in response['hits']['hits']:
            doc = Document(
                page_content=hit['_source'].get('content', ''),
                metadata={
                    'id': hit['_id'],
                    'score': hit['_score'],
                    'source': hit['_source'].get('source', 'desconhecido'),
                    'chunk_id': hit['_source'].get('chunk_id', 0)
                }
            )
            documents.append(doc)
        
        return documents

# Instanciar retriever
base_retriever = OpenSearchKNNRetriever(
    opensearch_client=os_client,
    embeddings_model=embeddings_model,
    index_name=INDEX_NAME,
    k=5
)

print('✅ Retriever OpenSearch pronto!')

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

# ── Configurar MultiQueryRetriever ────────────────────────────────────
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    prompt=PROMPT_MULTI_QUERY,
    include_original=True  # inclui a query original além das N variações
)

# Ativar logging para ver as variações geradas
logging.getLogger('langchain.retrievers.multi_query').setLevel(logging.INFO)

print('✅ MultiQueryRetriever configurado!')
print('Configuração:')
print(f'  - Inclui query original: True')
print(f'  - Variações por query: 4')
print(f'  - Docs por variação (k): 5')
print(f'  - Docs máximos (sem dedup): {5 * 5} (5 variações × k=5)')

## 6. Deduplicação por Similaridade de Coseno

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def deduplicate_by_cosine_similarity(
    docs: List[Document],
    embeddings_model,
    threshold: float = 0.85
) -> tuple[List[Document], dict]:
    """
    Remove documentos semanticamente duplicados usando similaridade de coseno.
    
    Parâmetros:
        docs: lista de documentos LangChain
        embeddings_model: modelo sentence-transformers
        threshold: limiar acima do qual documentos são considerados duplicatas
                   (0.85 = documentos com 85%+ de similaridade são removidos)
    
    Retorna:
        (docs_unicos, stats) onde stats contém métricas da deduplicação
    """
    if len(docs) <= 1:
        return docs, {'total': len(docs), 'removidos': 0, 'unicos': len(docs)}
    
    # Gerar embeddings para todos os documentos
    textos = [doc.page_content for doc in docs]
    embeddings = embeddings_model.encode(textos, normalize_embeddings=True)
    
    # Calcular matriz de similaridade (N × N)
    matriz_sim = cosine_similarity(embeddings)
    
    # Algoritmo greedy: seleciona documentos que não são muito similares
    # aos já selecionados
    selecionados = [0]  # sempre incluir o primeiro
    
    for i in range(1, len(docs)):
        # Verificar similaridade máxima com todos os já selecionados
        sim_max = max(matriz_sim[i][j] for j in selecionados)
        
        if sim_max < threshold:
            selecionados.append(i)
    
    docs_unicos = [docs[i] for i in selecionados]
    
    stats = {
        'total_antes': len(docs),
        'removidos': len(docs) - len(docs_unicos),
        'total_depois': len(docs_unicos),
        'threshold': threshold
    }
    
    return docs_unicos, stats

print('✅ Função de deduplicação definida!')
print(f'Threshold padrão: 0.85 (documentos com >85% de similaridade são removidos)')

## 7. Teste com Queries Jurídicas

In [ ]:
import time

# Queries jurídicas com diferentes graus de especificidade
queries_juridicas = [
    'Posso ser preso por não pagar uma dívida?',
    'Quais são os direitos do suspeito durante uma abordagem policial?',
    'Como funciona a prisão preventiva no processo penal brasileiro?'
]

resultados = []

for query in queries_juridicas:
    print(f'\n{'='*60}')
    print(f'QUERY: {query}')
    print('='*60)
    
    t0 = time.time()
    
    # Retrieval com Multi-Query (sem deduplicação do LangChain)
    docs_brutos = multi_query_retriever.get_relevant_documents(query)
    
    t_retrieval = time.time() - t0
    
    # Deduplicação por similaridade
    t0 = time.time()
    docs_dedupados, stats = deduplicate_by_cosine_similarity(
        docs_brutos, embeddings_model, threshold=0.85
    )
    t_dedup = time.time() - t0
    
    print(f'\n📊 Estatísticas:')
    print(f'  Documentos antes da deduplicação: {stats["total_antes"]}')
    print(f'  Documentos removidos (similares): {stats["removidos"]}')
    print(f'  Documentos únicos finais:          {stats["total_depois"]}')
    print(f'  Tempo de retrieval: {t_retrieval:.2f}s')
    print(f'  Tempo de deduplicação: {t_dedup:.3f}s')
    
    print(f'\n📄 Top 3 documentos únicos:')
    for i, doc in enumerate(docs_dedupados[:3], 1):
        fonte = doc.metadata.get('source', 'N/A')
        print(f'  {i}. [{fonte}] {doc.page_content[:100]}...')
    
    resultados.append({
        'query': query,
        'docs_brutos': stats['total_antes'],
        'docs_removidos': stats['removidos'],
        'docs_unicos': stats['total_depois'],
        'latencia_s': round(t_retrieval, 2)
    })

## 8. Tabela Comparativa

In [ ]:
# Criar DataFrame comparativo
df = pd.DataFrame(resultados)
df.columns = ['Query', 'Docs (bruto)', 'Removidos', 'Docs (único)', 'Latência (s)')

# Truncar queries para exibição
df['Query'] = df['Query'].apply(lambda x: x[:40] + '...' if len(x) > 40 else x)

print('\n=== TABELA COMPARATIVA: Documentos Antes vs Após Deduplicação ===')
print(df.to_string(index=False))

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Documentos antes vs depois
x = range(len(resultados))
width = 0.35
axes[0].bar([i - width/2 for i in x], df['Docs (bruto)'], width, label='Bruto', color='#e74c3c', alpha=0.8)
axes[0].bar([i + width/2 for i in x], df['Docs (único)'], width, label='Deduplicado', color='#2ecc71', alpha=0.8)
axes[0].set_title('Documentos: Antes vs Após Deduplicação\n(threshold=0.85)', fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Q{i+1}' for i in x])
axes[0].set_ylabel('Número de Documentos')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Gráfico 2: Percentual removido
pct_removidos = [r['docs_removidos'] / r['docs_brutos'] * 100 for r in resultados]
axes[1].bar(x, pct_removidos, color='#e67e22', alpha=0.8)
axes[1].set_title('Percentual de Documentos\nRemovidos por Deduplicação', fontsize=12)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Q{i+1}' for i in x])
axes[1].set_ylabel('% Removidos')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(pct_removidos):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('multi_query_deduplicacao.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Gráfico salvo como multi_query_deduplicacao.png')

## 9. Experimento: O Que Acontece com N=8?

In [ ]:
# ── Comparação entre N=4 (padrão) e N=8 (excessivo) ──────────────────

PROMPT_N8 = PromptTemplate(
    input_variables=['question'],
    template="""Gere exatamente 8 versões alternativas da pergunta abaixo.
Retorne APENAS as perguntas, uma por linha.

Pergunta: {question}

Versões:"""
)

query_teste = 'Quais são os direitos do suspeito em uma abordagem policial?'

print(f'Query: {query_teste}')
print('\n--- Gerando 8 variações ---')

response_n8 = llm.invoke(PROMPT_N8.format(question=query_teste))
variacoes_n8 = [v.strip() for v in response_n8.content.strip().split('\n') if v.strip()]

print(f'\nVariações geradas ({len(variacoes_n8)}):')
for i, v in enumerate(variacoes_n8, 1):
    print(f'  {i}. {v}')

# Análise: as últimas variações tendem a se afastar da query original?
print('\n--- Análise de Drift Semântico ---')
query_emb = embeddings_model.encode(query_teste, normalize_embeddings=True)

for i, variacao in enumerate(variacoes_n8, 1):
    var_emb = embeddings_model.encode(variacao, normalize_embeddings=True)
    sim = np.dot(query_emb, var_emb)
    qualidade = '✅ Boa' if sim > 0.75 else ('⚠️ Moderada' if sim > 0.60 else '❌ Drift')
    print(f'  Var {i}: sim={sim:.3f} | {qualidade} | {variacao[:60]}...')

print('\n💡 Observação: variações acima de N=5 frequentemente apresentam drift')
print('   semântico (similaridade < 0.70), recuperando documentos irrelevantes.')

## 10. Conclusões e Reflexão

Responda as questões abaixo em células de texto (Markdown):

### Questão 1
Com base nos resultados obtidos, qual foi a porcentagem média de documentos removidos pela deduplicação? Isso indica que o Multi-Query estava recuperando muitos duplicados ou poucos?

> **Sua resposta aqui:**

### Questão 2
No experimento com N=8, a partir de qual variação você observou query drift? Qual threshold de similaridade você usaria para filtrar automaticamente variações ruins?

> **Sua resposta aqui:**

### Questão 3
Para um sistema de busca jurídica em produção (alto volume, 10.000 queries/dia), você usaria N=4 ou N=3? Justifique considerando custo, latência e ganho de recall.

> **Sua resposta aqui:**

In [ ]:
# ── Checklist de Entrega ──────────────────────────────────────────────
print('=== CHECKLIST DE ENTREGA — LAB 1 ===')
checklist = [
    'MultiQueryRetriever configurado com vLLM e prompt jurídico customizado',
    '4 variações geradas para 3 queries jurídicas distintas',
    'Deduplicação por similaridade de coseno implementada (threshold=0.85)',
    'Tabela comparativa documentada (antes/depois da deduplicação)',
    'Experimento N=8 executado com análise de drift semântico',
    'Questões de reflexão respondidas'
]

for item in checklist:
    print(f'  [ ] {item}')

print('\n✅ Lab 1 concluído! Prossiga para o LAB 2 — Step-Back Prompting.')